# Xuất số liệu và kiểm tra kết quả (chỉ đọc, chạy trên runtime CPU)

Notebook này **không train gì**. Nó đọc `result.json` và `preds_test.npz` trên Drive để:
1. Lập danh mục mọi run (Sleep-EDF và HMC): xong, thiếu, đang dở.
2. Lập bảng **val Macro-F1** theo nhánh (dùng để chốt Teacher chính và Edge nạp lên ESP32 — **không dùng số test**).
3. Kiểm tra hội tụ (run chạm số epoch tối đa, model đoán dồn vào một lớp).
4. Kiểm tra nhất quán (mọi nhánh cùng tập epoch test; số epoch test mỗi fold khớp `AUDITED_NUMBERS`).
5. Đóng gói `results_export.zip` (không gồm checkpoint `.pt`) để đưa về project — mọi con số trong bài sẽ được script tính lại từ gói này.

In [ ]:
import os, json, glob, hashlib, zipfile
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
DRIVE_ROOT = os.environ.get('SKD_ROOT', '/content/drive/MyDrive/sleep_kd_r2')
RESULTS = f'{DRIVE_ROOT}/results'
os.makedirs(RESULTS, exist_ok=True)
DATASETS = {'SleepEDF': f'{DRIVE_ROOT}/runs', 'HMC': f'{DRIVE_ROOT}/runs_hmc'}
STAGES = ['W', 'N1', 'N2', 'N3', 'REM']
MAX_EPOCHS = 80
# số epoch test mỗi fold theo AUDITED_NUMBERS (tính độc lập từ dữ liệu gốc)
EXPECTED_TEST = {'SleepEDF': [53332, 48522, 47935, 44604, 43543], 'HMC': [28476, 27492, 27644, 25808, 26591]}
print({k: os.path.isdir(v) for k, v in DATASETS.items()})

## 1. Danh mục run

In [ ]:
rows, in_progress = [], []
for ds, root in DATASETS.items():
    for d in sorted(glob.glob(f'{root}/seed*/*/fold*')):
        parts = os.path.normpath(d).split(os.sep)
        seed, arm, fold = parts[-3][4:], parts[-2], int(parts[-1][4:])
        rp = f'{d}/result.json'
        if not os.path.exists(rp):
            if os.path.exists(f'{d}/last.pt'):
                in_progress.append((ds, arm, fold))
            continue
        r = json.load(open(rp)); t = r['test']
        rows.append(dict(dataset=ds, seed=int(seed), arm=arm, fold=fold, kind=r['kind'], loss=r['loss'], src=r['src'],
                         T=r.get('T'), lr=r['lr'], ls=r.get('ls'), bs=r['bs'], selected_from=r.get('selected_from'),
                         best_val_mf1=r['best_val_mf1'], best_epoch=r['best_epoch'], epochs_run=r['epochs_run'],
                         minutes=round(r['minutes'], 1), n_train=r['n_train'], n_val=r['n_val'], n_test=r['n_test'],
                         params_inference=r['params_inference'], test_acc=t['acc'], test_mf1=t['mf1'],
                         test_kappa=t['kappa'], test_n1=t['f1'][1]))
inv = pd.DataFrame(rows)
inv.to_csv(f'{RESULTS}/run_inventory.csv', index=False)
print('runs done:', len(inv), '| in progress (có last.pt):', in_progress)
cov = inv.pivot_table(index=['dataset', 'arm'], columns='fold', values='seed', aggfunc='count').fillna(0).astype(int)
cov

## 2. Validation Macro-F1 theo nhánh (dùng để chốt lựa chọn — không dùng test)
Cột `mean_folds1_4` loại fold 0 vì ở fold 0 `T_ce` là cấu hình được chọn từ lưới tune trên chính tập validation đó (val bị đẩy cao do chọn lọc).

In [ ]:
v = inv[~inv.arm.str.startswith('Ttune_') & (inv.seed == 42)].pivot_table(index=['dataset', 'kind', 'arm'], columns='fold', values='best_val_mf1')
v['mean_5'] = v[[c for c in v.columns if isinstance(c, (int, np.integer))]].mean(1)
v['mean_folds1_4'] = v[[c for c in v.columns if isinstance(c, (int, np.integer)) and c > 0]].mean(1)
v['n_folds'] = v[[c for c in v.columns if isinstance(c, (int, np.integer))]].notna().sum(1)
v = (v * [100 if isinstance(c, (int, np.integer)) or c.startswith('mean') else 1 for c in v.columns]).round(2)
v.to_csv(f'{RESULTS}/validation_mf1_by_arm.csv')
display(v)
tune = inv[inv.arm.str.startswith('Ttune_')][['dataset', 'arm', 'lr', 'ls', 'best_val_mf1', 'best_epoch', 'epochs_run']]
tune.sort_values('best_val_mf1', ascending=False)

## 3. Kiểm tra hội tụ và dự đoán

In [ ]:
# (a) run chạm số epoch tối đa mà epoch tốt nhất nằm ở cuối -> có thể thiếu ngân sách epoch
cap = inv[(inv.epochs_run >= MAX_EPOCHS) & (inv.best_epoch >= MAX_EPOCHS - 10)]
print('run chạm trần epoch và vẫn đang cải thiện:', len(cap)); display(cap[['dataset', 'arm', 'fold', 'best_epoch', 'epochs_run']])
# (b) phân bố lớp dự đoán trên test: phát hiện model đoán dồn vào một lớp
pr = []
for ds, root in DATASETS.items():
    for _, r in inv[inv.dataset == ds].iterrows():
        p = f'{root}/seed{r.seed}/{r.arm}/fold{r.fold}/preds_test.npz'
        d = np.load(p)
        pred = d['logits'].astype(np.float32).argmax(1)
        frac = np.bincount(pred, minlength=5) / len(pred)
        true = np.bincount(d['y'].astype(int), minlength=5) / len(d['y'])
        pr.append(dict(dataset=ds, arm=r.arm, fold=r.fold, **{f'pred_{s}': round(100 * f, 2) for s, f in zip(STAGES, frac)},
                       **{f'true_{s}': round(100 * f, 2) for s, f in zip(STAGES, true)},
                       min_pred_pct=round(100 * frac.min(), 2), nan_logits=bool(np.isnan(d['logits'].astype(np.float32)).any())))
pr = pd.DataFrame(pr); pr.to_csv(f'{RESULTS}/prediction_distribution.csv', index=False)
print('run có lớp dự đoán < 1%:', int((pr.min_pred_pct < 1).sum()), '| run có NaN:', int(pr.nan_logits.sum()))
pr.sort_values('min_pred_pct').head(10)

## 4. Kiểm tra nhất quán

In [ ]:
issues = []
for ds, root in DATASETS.items():
    sub = inv[inv.dataset == ds]
    for (seed, fold), g in sub.groupby(['seed', 'fold']):
        ref = None
        for _, r in g.iterrows():
            d = np.load(f'{root}/seed{seed}/{r.arm}/fold{fold}/preds_test.npz')
            key = (d['rec'].tolist(), d['center'].tolist(), d['y'].tolist())
            if ref is None:
                ref, ref_arm = key, r.arm
            elif key != ref:
                issues.append(f'{ds} seed{seed} fold{fold}: {r.arm} test epochs differ from {ref_arm}')
        exp = EXPECTED_TEST[ds][fold]
        bad = g[g.n_test != exp]
        if len(bad):
            issues.append(f'{ds} fold{fold}: n_test {sorted(set(bad.n_test))} != expected {exp} ({sorted(bad.arm)})')
print('OK — mọi nhánh cùng tập epoch test và khớp AUDITED_NUMBERS' if not issues else '\n'.join(issues))

## 5. Đóng gói để đưa về project

In [ ]:
out = f'{DRIVE_ROOT}/results_export.zip'
n = 0
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for ds, root in DATASETS.items():
        for p in glob.glob(f'{root}/**/*', recursive=True):
            if os.path.isfile(p) and (p.endswith(('result.json', 'preds_test.npz')) or os.path.basename(p).startswith('cross_')):
                z.write(p, os.path.relpath(p, DRIVE_ROOT)); n += 1
    for p in glob.glob(f'{RESULTS}/*') + glob.glob(f'{DRIVE_ROOT}/*.csv') + glob.glob(f'{DRIVE_ROOT}/*.json'):
        if os.path.isfile(p):
            z.write(p, os.path.relpath(p, DRIVE_ROOT)); n += 1
sha = hashlib.sha256(open(out, 'rb').read()).hexdigest()
print(f'{n} files | {os.path.getsize(out) / 1e6:.1f} MB | sha256 {sha}')
print('Tải file này về và đặt vào revision/R2_subject_level/ ; gửi kèm dòng sha256 ở trên.')

## 6. Đóng gói checkpoint cho Zenodo (chỉ `best.pt`, không lấy `last.pt`)

In [ ]:
out = f'{DRIVE_ROOT}/checkpoints_export.zip'
n = 0
with zipfile.ZipFile(out, 'w', zipfile.ZIP_STORED) as z:          # .pt is already compressed
    for ds, root in DATASETS.items():
        for p in sorted(glob.glob(f'{root}/seed*/*/fold*/best.pt')):
            z.write(p, 'checkpoints/' + os.path.relpath(p, DRIVE_ROOT)); n += 1
sha = hashlib.sha256(open(out, 'rb').read()).hexdigest()
print(f'{n} checkpoints | {os.path.getsize(out) / 1e6:.1f} MB | sha256 {sha}')